# hklforge vs diffcalc, systematically

diffcalc-core 0.4 is the oracle for psic: for every constraint combination it
implements (194), random crystals, wavelengths, reflections and values, the
**set** of positions must be the same. This is a compact version of
`tests/test_solve_diffcalc.py` / `dev/diffcalc_stats.py`; the full tally is in
[docs/verification.md](../docs/verification.md).

In [1]:
import sys, os, logging, warnings
sys.path.insert(0, os.path.abspath("../src"))      # hklforge from this repository
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hklforge import Diffractometer, Geometry, ModeError, Degenerate
from hklforge.lattice import b_matrix
from hklforge.draw import draw, draw_circles
logging.getLogger("diffcalc").setLevel(logging.ERROR)
pd.set_option("display.precision", 3)
%matplotlib inline

In [2]:
from diffcalc.ub.calc import UBCalculation
from diffcalc.hkl.calc import HklCalculation
from diffcalc.hkl.constraints import Constraints
from diffcalc.hkl.geometry import Position
from diffcalc.util import DiffcalcException

import contextlib, io

def diffcalc_ubcalc(lattice, U=np.eye(3), surface=None, reference=None):
    ubc = UBCalculation("example")
    ubc.set_lattice("x", *lattice)
    with contextlib.redirect_stdout(io.StringIO()):     # diffcalc prints "Calculating UB matrix."
        ubc.set_u(U)
    if surface is not None:
        ubc.surf_nphi = tuple(np.asarray(surface) / np.linalg.norm(surface))
    if reference is not None:
        ubc.n_phi = tuple(np.asarray(reference) / np.linalg.norm(reference))
    return ubc

def diffcalc_solve(ubc, hkl, mode, wavelength):
    """diffcalc's positions as dicts, [] if it finds none, or the error text."""
    try:
        return [p.asdict for p, _ in HklCalculation(ubc, Constraints(mode)).get_position(*hkl, wavelength)]
    except DiffcalcException as e:
        msg = " ".join(str(e).replace("*", " ").split())
        if "No solutions" in msg or "prohibits a solution" in msg:
            return []
        return msg
    except (AssertionError, ValueError) as e:
        return f"{type(e).__name__}: {e}"

def same(a, b, tol=1e-5):
    return all(abs((a[n] - b[n] + 180) % 360 - 180) < tol for n in a)

def same_set(xs, ys):
    return all(any(same(x, y) for y in ys) for x in xs) and all(any(same(y, x) for x in xs) for y in ys)

In [3]:
import itertools, collections, time
psic = Geometry.builtin("psic")
SAMPLE = ("mu", "eta", "chi", "phi")
DETECTOR = ("delta", "nu", "qaz", "naz")
REFS = ("betain", "betaout", "bin_eq_bout", "alpha", "beta", "a_eq_b", "psi")
BISECT = (("mu", "bisect"), ("eta", "bisect"), ("omega", "bisect"))
MODES = (list(itertools.combinations(SAMPLE, 3))
         + [(d_,) + s for d_ in DETECTOR for s in list(itertools.combinations(SAMPLE, 2)) + list(BISECT)]
         + [s + (r,) for s in itertools.combinations(SAMPLE, 2) for r in REFS]
         + [(d_, r, s) for d_ in DETECTOR for r in REFS for s in SAMPLE])
print(len(MODES), "combinations")

def value(name, rng):
    if name in ("bin_eq_bout", "a_eq_b", "bisect"): return True
    if name == "omega": return float(rng.uniform(-30, 30))
    if name in ("psi", "naz", "qaz"): return float(rng.uniform(-180, 180))
    if name in REFS: return float(rng.uniform(-20, 40))
    return float(rng.uniform(-170, 170))

def case(names, seed):
    rng = np.random.default_rng(seed)
    lattice = (*rng.uniform(3, 8, 3), *rng.uniform(70, 110, 3))
    q, _ = np.linalg.qr(rng.normal(size=(3, 3))); U = q * np.sign(np.linalg.det(q))
    surf, ref = (v / np.linalg.norm(v) for v in rng.normal(size=(2, 3)))
    lam = rng.uniform(0.7, 2.0)
    ub = U @ b_matrix(*lattice)
    while True:
        hkl = tuple(int(x) for x in rng.integers(-3, 4, 3))
        if 0 < np.linalg.norm(ub @ hkl) < 1.8 * 2 * np.pi / lam: break
    mode = {n: value(n, rng) for n in names}
    if "naz" in mode and set(mode) & {"betain", "betaout", "bin_eq_bout"}:
        ref = surf                         # see diffcalc bug 4 below
    return lattice, U, surf, ref, lam, hkl, mode

194 combinations


In [4]:
from hklforge.virtual import bisect_axis

def violates(d, a, mode):
    """How far a position is from satisfying hkl-independent parts of the mode."""
    err = 0.0
    if "bisect" in mode:
        e, Q = bisect_axis(psic, a), psic.q_lab(a, 1.0)
        err = abs(e @ np.cross(psic.beam, Q / np.linalg.norm(Q)))
        if "omega" in mode:
            err = max(err, np.radians(abs((d.pseudo(a)["omega"] - mode["omega"] + 180) % 360 - 180)))
    return err

rows = []
t_mine = t_theirs = 0.0
for names in MODES:
    for seed in range(3):
        lattice, U, surf, ref, lam, hkl, mode = case(names, seed)
        d = Diffractometer(psic, U @ b_matrix(*lattice), wavelength=lam, surface=surf, reference=ref)
        ubc = diffcalc_ubcalc(lattice, U, surf, ref)
        row = {"mode": "-".join(names), "seed": seed}
        try:
            t = time.perf_counter(); mine = d.solve(hkl, mode); t_mine += time.perf_counter() - t
        except ModeError as e:
            rows.append(row | {"result": "hklforge: mode not supported"}); continue
        t = time.perf_counter(); theirs = diffcalc_solve(ubc, hkl, mode, lam); t_theirs += time.perf_counter() - t
        if isinstance(theirs, str):
            fwd = HklCalculation(ubc, Constraints({}))
            ok = all(np.allclose(fwd.get_hkl(Position(**a), lam), hkl, atol=1e-9) for a in mine)
            kind = "no code" if "No code yet" in theirs else "diffcalc failed"
            rows.append(row | {"result": kind, "hklforge": len(mine), "hklforge ok by diffcalc get_hkl": ok})
            continue
        bad = [a for a in theirs if violates(d, a, mode) > 1e-6]
        good = [a for a in theirs if a not in bad]
        rows.append(row | {"result": "identical" if same_set(mine, good) else "DIFFERENT",
                           "hklforge": len(mine), "diffcalc": len(theirs), "diffcalc violating mode": len(bad)})
df = pd.DataFrame(rows)
print(f"hklforge {t_mine:.1f} s, diffcalc {t_theirs:.1f} s in total")
df.groupby("result").size()

hklforge 2.0 s, diffcalc 1.2 s in total


result
diffcalc failed                   7
hklforge: mode not supported      9
identical                       548
no code                          18
dtype: int64

Per mode family (with solutions only):

In [5]:
def family(m):
    n = m.split("-")
    if "bisect" in n: return f"{n[0]} + {n[1]} + bisect"
    kinds = ["det" if x in ("delta", "nu") else x if x in ("qaz", "naz", "psi") else "ref" if x in REFS else "sample" for x in n]
    return " + ".join(f"{kinds.count(k)} {k}" for k in dict.fromkeys(kinds))
ok = df[df.result == "identical"].assign(family=lambda t: t["mode"].map(family))
ok.groupby("family").agg(cases=("mode", "size"), positions=("hklforge", "sum"),
                         max_branches=("hklforge", "max"), diffcalc_violating=("diffcalc violating mode", "sum"))

,cases,positions,max_branches,diffcalc_violating
family,,,,
1 det + 1 psi + 1 sample,24,58.0,4.0,0.0
1 det + 1 ref + 1 sample,144,642.0,8.0,0.0
1 det + 2 sample,34,62.0,4.0,0.0
1 naz + 1 psi + 1 sample,12,44.0,4.0,0.0
1 naz + 1 ref + 1 sample,72,468.0,8.0,0.0
1 qaz + 1 psi + 1 sample,12,40.0,4.0,0.0
1 qaz + 1 ref + 1 sample,72,500.0,8.0,0.0
1 qaz + 2 sample,17,44.0,4.0,0.0
2 sample + 1 psi,18,52.0,4.0,0.0


## The diffcalc bugs this found

### 1. eta, phi and one detector circle held, |eta| > 90°
diffcalc returns only wrong positions, then its own self-check raises.

In [6]:
lattice, U = (5.0, 5.5, 6.0, 90, 95, 100), np.eye(3)
ub = U @ b_matrix(*lattice); lam = 1.2
d = Diffractometer(psic, ub, wavelength=lam); ubc = diffcalc_ubcalc(lattice, U)
for eta in (40, 140):
    m = {"nu": 10, "eta": eta, "phi": 30}
    calc = HklCalculation(ubc, Constraints(m))
    calc._HklCalculation__verify_pos_map_to_hkl = lambda *a, **k: None    # see what it would return
    theirs = [p.asdict for p, _ in calc.get_position(1, 2, 2, lam)]
    err = max(np.abs(np.array(HklCalculation(ubc, Constraints({})).get_hkl(Position(**a), lam)) - (1, 2, 2)).max() for a in theirs)
    mine = d.solve((1, 2, 2), m)
    print(f"eta = {eta}: diffcalc {len(theirs)} positions, worst hkl error {err:.2g}; hklforge {len(mine)}, "
          f"worst error {max(np.abs(d.hkl(a) - (1, 2, 2)).max() for a in mine):.1g}")

eta = 40: diffcalc 2 positions, worst hkl error 2.7e-15; hklforge 2, worst error 2e-15
eta = 140: diffcalc 2 positions, worst hkl error 3.4; hklforge 2, worst error 2e-15


### 2. bisect: extra positions that violate the mode

In [7]:
bis = df[df["mode"].str.contains("bisect") & (df.result == "identical")]
print(f"{int(bis['diffcalc violating mode'].sum())} of {int(bis['diffcalc'].sum())} diffcalc positions in "
      f"{len(bis)} bisect cases violate bisect or omega; the rest equal hklforge's")

120 of 212 diffcalc positions in 23 bisect cases violate bisect or omega; the rest equal hklforge's


### 3. eta + bisect: diffcalc crashes

In [8]:
df[df.result == "diffcalc failed"]

,mode,seed,result,hklforge,diffcalc,diffcalc violating mode,hklforge ok by diffcalc get_hkl
26,delta-eta-phi,2,diffcalc failed,4.0,NaN,NaN,True
34,delta-eta-bisect,1,diffcalc failed,0.0,NaN,NaN,True
53,nu-eta-phi,2,diffcalc failed,4.0,NaN,NaN,True
61,nu-eta-bisect,1,diffcalc failed,0.0,NaN,NaN,True
62,nu-eta-bisect,2,diffcalc failed,0.0,NaN,NaN,True
80,qaz-eta-phi,2,diffcalc failed,4.0,NaN,NaN,True
88,qaz-eta-bisect,1,diffcalc failed,0.0,NaN,NaN,True


### 4. naz with a surface constraint
diffcalc solves naz for the surface normal, then filters by naz of the reference vector:

In [9]:
lattice, U = (4.0, 4.0, 4.0, 90, 90, 90), np.eye(3)
ub = b_matrix(*lattice); lam = 1.0
m = {"naz": 40, "betain": 5, "mu": 0}
for ref in ((0, 1, 1), (0, 0, 1)):
    ubc = diffcalc_ubcalc(lattice, U, surface=(0, 0, 1), reference=ref)
    d = Diffractometer(psic, ub, wavelength=lam, surface=(0, 0, 1), reference=(0, 0, 1))
    theirs = diffcalc_solve(ubc, (1, 1, 2), m, lam)
    print(f"diffcalc reference vector {ref}: diffcalc {theirs if isinstance(theirs, str) else len(theirs)}, "
          f"hklforge (naz of the surface normal) {len(d.solve((1, 1, 2), m))}")

diffcalc reference vector (0, 1, 1): diffcalc 0, hklforge (naz of the surface normal) 8
diffcalc reference vector (0, 0, 1): diffcalc 8, hklforge (naz of the surface normal) 8


### 5. naz with two sample circles: "No code yet"
diffcalc lists the mode as implemented, but has no solver for it. hklforge's answer is
checked against a brute-force scan instead (`dev/check_naz_scan.py`):

In [10]:
sys.path.insert(0, os.path.abspath("../dev"))
import check_naz_scan
check_naz_scan.main(10);

naz + 2 held sample circles, psic, sample settings (detector follows): {'cases': 7, 'scan': 10, 'hklforge': 10, 'scan_not_in_hklforge': 0, 'hklforge_not_in_scan': 0}
